# Interactive slab-waveguide dispersion explorer

This notebook accompanies the **Photonic Devices 2026** course.

It explores the dependence of the effective index of a symmetric dielectric slab waveguide on wavelength and slab thickness:

$$n_{\rm eff}=n_{\rm eff}(\lambda,d).$$

Interactive controls allow you to change the core and cladding indices, polarization, mode order, slab thickness, and wavelength.

**Important:** in this notebook $n_1$ and $n_2$ are assumed independent of wavelength. Any variation of $n_{\rm eff}$ with $\lambda$ is therefore **pure waveguide dispersion**.


## 1. Normalized description

For a symmetric slab of total thickness $d$, with $a=d/2$,

$$V=k_0a\sqrt{n_1^2-n_2^2}
=\frac{\pi d}{\lambda}\sqrt{n_1^2-n_2^2}.$$

For fixed $n_1$, $n_2$, polarization, and mode order, the modal solution depends on geometry and wavelength through $V$.

The effective index satisfies

$$n_2<n_{\rm eff}<n_1$$

for a guided mode, while a higher-order mode disappears when its cutoff is crossed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass


## 2. Modal solver

The cell below solves one selected TE or TM mode in normalized form. You normally do not need to modify it.


In [ ]:
def neff_from_V(V, n1, n2, polarization='TE', mode_order=0):
    """Return n_eff for one symmetric-slab mode at normalized frequency V.

    mode_order = 0,1,2,... corresponds to TE0/TM0, TE1/TM1, ...
    Returns NaN below cutoff.
    """
    if n1 <= n2 or V <= 0:
        return np.nan

    m = int(mode_order)
    Vc = m * np.pi / 2

    if V <= Vc + 1e-10:
        return np.nan

    rho = 1.0 if polarization == 'TE' else (n1 / n2)**2
    eps = 1e-8

    u_left = Vc + eps
    u_right = min((m + 1) * np.pi / 2 - eps, V - eps)

    if u_right <= u_left:
        return np.nan

    def w(u):
        return np.sqrt(max(V*V - u*u, 0.0))

    if m % 2 == 0:
        def f(u):
            return u * np.tan(u) - rho * w(u)
    else:
        def f(u):
            return -u / np.tan(u) - rho * w(u)

    try:
        u = brentq(f, u_left, u_right, maxiter=200)
    except ValueError:
        return np.nan

    # b = (n_eff^2 - n2^2)/(n1^2 - n2^2)
    b = 1.0 - (u / V)**2
    b = np.clip(b, 0.0, 1.0)
    return np.sqrt(n2**2 + b * (n1**2 - n2**2))


def make_neff_lookup(n1, n2, polarization, mode_order, Vmax, npts=700):
    Vc = mode_order * np.pi / 2
    Vmin = max(Vc + 1e-5, 1e-5)

    if Vmax <= Vmin:
        return np.array([Vmin, Vmax]), np.array([np.nan, np.nan])

    Vgrid = np.linspace(Vmin, Vmax, npts)
    neff_grid = np.array([
        neff_from_V(V, n1, n2, polarization, mode_order)
        for V in Vgrid
    ])
    return Vgrid, neff_grid


def interp_neff(Vvalues, Vgrid, neff_grid, Vc):
    Vvalues = np.asarray(Vvalues)
    out = np.full(Vvalues.shape, np.nan, dtype=float)
    valid = (Vvalues > Vc) & (Vvalues >= Vgrid[0]) & (Vvalues <= Vgrid[-1])
    if np.any(valid):
        out[valid] = np.interp(Vvalues[valid], Vgrid, neff_grid)
    return out


## 3. Interactive dispersion explorer

The heat map shows $n_{\rm eff}(\lambda,d)$ for the selected mode. The two following plots are cuts through the same surface at the currently selected thickness and wavelength.


In [ ]:
def update_dispersion(n1, n2, d_um, wavelength_um, polarization, mode_order):
    if n1 <= n2:
        print('Guiding condition not satisfied: choose n1 > n2.')
        return

    # Fixed plotting ranges for the explorer
    lambda_min, lambda_max = 0.50, 2.50   # um
    d_min, d_max = 0.10, 3.00             # um

    lambda_grid = np.linspace(lambda_min, lambda_max, 120)
    d_grid = np.linspace(d_min, d_max, 120)

    index_factor = np.sqrt(n1**2 - n2**2)
    Vmax = np.pi * d_max / lambda_min * index_factor
    Vc = mode_order * np.pi / 2

    Vgrid, neff_lookup = make_neff_lookup(
        n1, n2, polarization, mode_order, Vmax
    )

    # ------------------------------------------------------------
    # 2D map n_eff(lambda,d)
    # ------------------------------------------------------------
    L, D = np.meshgrid(lambda_grid, d_grid)
    Vmap = np.pi * D / L * index_factor
    neff_map = interp_neff(Vmap, Vgrid, neff_lookup, Vc)

    plt.figure(figsize=(8.2, 5.2))
    pcm = plt.pcolormesh(lambda_grid, d_grid, neff_map, shading='auto')
    plt.colorbar(pcm, label=r'$n_{\rm eff}$')
    plt.scatter([wavelength_um], [d_um], s=55, label='selected point')
    plt.xlabel(r'Wavelength $\lambda$ ($\mu$m)')
    plt.ylabel(r'Slab thickness $d$ ($\mu$m)')
    plt.title(f'{polarization}{mode_order}: effective-index map')
    plt.legend()
    plt.show()

    # ------------------------------------------------------------
    # Cut 1: n_eff(lambda) at selected d
    # ------------------------------------------------------------
    V_lambda = np.pi * d_um / lambda_grid * index_factor
    neff_lambda = interp_neff(V_lambda, Vgrid, neff_lookup, Vc)

    V_selected = np.pi * d_um / wavelength_um * index_factor
    neff_selected = neff_from_V(
        V_selected, n1, n2, polarization, mode_order
    )

    plt.figure(figsize=(8.2, 4.3))
    plt.plot(lambda_grid, neff_lambda)
    if np.isfinite(neff_selected):
        plt.scatter([wavelength_um], [neff_selected], s=55)
    plt.axhline(n1, linestyle=':', label=r'$n_1$')
    plt.axhline(n2, linestyle=':', label=r'$n_2$')
    plt.xlabel(r'Wavelength $\lambda$ ($\mu$m)')
    plt.ylabel(r'$n_{\rm eff}$')
    plt.title(f'{polarization}{mode_order}: $n_{{eff}}(\lambda)$ at d = {d_um:.2f} um')
    plt.grid(True)
    plt.legend()
    plt.show()

    # ------------------------------------------------------------
    # Cut 2: n_eff(d) at selected lambda
    # ------------------------------------------------------------
    V_d = np.pi * d_grid / wavelength_um * index_factor
    neff_d = interp_neff(V_d, Vgrid, neff_lookup, Vc)

    plt.figure(figsize=(8.2, 4.3))
    plt.plot(d_grid, neff_d)
    if np.isfinite(neff_selected):
        plt.scatter([d_um], [neff_selected], s=55)
    plt.axhline(n1, linestyle=':', label=r'$n_1$')
    plt.axhline(n2, linestyle=':', label=r'$n_2$')
    plt.xlabel(r'Slab thickness $d$ ($\mu$m)')
    plt.ylabel(r'$n_{\rm eff}$')
    plt.title(f'{polarization}{mode_order}: n_eff(d) at λ = {wavelength_um:.2f} um')
    plt.grid(True)
    plt.legend()
    plt.show()

    # ------------------------------------------------------------
    # Numerical summary
    # ------------------------------------------------------------
    print(f'Polarization / mode: {polarization}{mode_order}')
    print(f'n1 = {n1:.3f}, n2 = {n2:.3f}')
    print(f'd = {d_um:.3f} um, lambda = {wavelength_um:.3f} um')
    print(f'V = {V_selected:.4f}')

    if np.isfinite(neff_selected):
        print(f'n_eff = {neff_selected:.6f}')
    else:
        print('Selected mode is below cutoff at this point.')


n1_slider = widgets.FloatSlider(
    value=1.50, min=1.05, max=4.00, step=0.05,
    description='n1:', continuous_update=False,
    style={'description_width': '40px'}, layout=widgets.Layout(width='350px'))

n2_slider = widgets.FloatSlider(
    value=1.00, min=1.00, max=3.00, step=0.05,
    description='n2:', continuous_update=False,
    style={'description_width': '40px'}, layout=widgets.Layout(width='350px'))

d_slider = widgets.FloatSlider(
    value=0.80, min=0.10, max=3.00, step=0.05,
    description='d (um):', continuous_update=False,
    style={'description_width': '70px'}, layout=widgets.Layout(width='350px'))

lambda_slider = widgets.FloatSlider(
    value=1.55, min=0.50, max=2.50, step=0.05,
    description='lambda (um):', continuous_update=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='350px'))

pol_selector = widgets.ToggleButtons(
    options=['TE', 'TM'], value='TE', description='Polarization:')

mode_selector = widgets.IntSlider(
    value=0, min=0, max=5, step=1,
    description='Mode order:', continuous_update=False,
    style={'description_width': '90px'}, layout=widgets.Layout(width='350px'))

controls = widgets.VBox([
    widgets.HBox([n1_slider, n2_slider]),
    widgets.HBox([d_slider, lambda_slider]),
    widgets.HBox([pol_selector, mode_selector])
])

interactive_output = widgets.interactive_output(
    update_dispersion,
    {
        'n1': n1_slider,
        'n2': n2_slider,
        'd_um': d_slider,
        'wavelength_um': lambda_slider,
        'polarization': pol_selector,
        'mode_order': mode_selector
    }
)

display(controls, interactive_output)


## 4. Suggested exploration

Try to predict each trend before moving the sliders.

1. At fixed $d$, increase $\lambda$. Why does $n_{\rm eff}$ generally move toward $n_2$?
2. At fixed $\lambda$, increase $d$. Why does confinement increase?
3. Reduce the index contrast $n_1-n_2$. How does the $n_{\rm eff}(\lambda,d)$ map change?
4. Compare TE and TM modes with the same order.
5. Select a higher-order mode and identify its cutoff boundary in the $(\lambda,d)$ plane.

Because $n_1$ and $n_2$ are constant in this notebook, the wavelength dependence of $n_{\rm eff}$ is generated entirely by the wavelength-dependent modal confinement.
